# 03 — Training

Training configuration and entry point (`backend/scripts/train.py`). Training itself is **not re-run inside this notebook** (BERT training alone takes real GPU/CPU time) -- this shows the exact, real code path and the hyperparameters actually used to produce the shipped checkpoints in `models/`. See `04_Evaluation.ipynb` for the resulting metrics.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from wherever this notebook actually lives to find the real
    project root (the folder containing both backend/app/ and data/), so every
    relative path used below resolves correctly regardless of which folder
    this notebook is opened from."""
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the Baseera project root (a folder containing both "
        "backend/app/ and data/) above this notebook's location."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


## Entry point

```bash
python backend/scripts/train.py --model bert  --epochs 3  --batch-size 8  --learning-rate 2e-5
python backend/scripts/train.py --model cnn2d --epochs 10 --batch-size 64 --learning-rate 0.001
```

In [ ]:
import inspect
from backend.scripts import train as train_script

print(inspect.getsource(train_script.build_split))


## Hyperparameters actually used (from README.md, matching the shipped checkpoints)

**BERT**: `max_len=128`, `batch_size=8`, `epochs=3`, `lr=2e-5`, `weight_decay=0.01`, linear warmup (10%) + decay, gradient clipping at 1.0, early stopping (patience 2), seed 42.

**CNN2D**: `batch_size=64`, `epochs=10`, `lr=1e-3`, `weight_decay=1e-3`, label smoothing 0.1, `ReduceLROnPlateau`, early stopping (patience 3), seed 42.

Both: dataset split, class-weight computation, and all RNG seeded at **42** (`app/ml/utils.py::set_seed`) -- see `08_Utility_Files.ipynb`.

## Reproducibility guarantee

Tokenizers are fit on the TRAIN partition only. The split manifest (`artifacts/split_manifest.json`) stores stable `review_id`/`text_hash` identifiers so evaluation never depends on re-running the split -- verified in `04_Evaluation.ipynb` and `backend/app/tests/test_preprocessing.py::test_split_has_no_text_overlap`.

In [ ]:
import json

manifest = json.load(open("artifacts/split_manifest.json", encoding="utf-8"))
print("Split manifest keys:", list(manifest.keys())[:10] if isinstance(manifest, dict) else type(manifest))
